# Variant Analysis Summary

This notebook generates visualizations for variant calling results:
- Variant type distribution
- Quality score distribution
- Transition/Transversion ratio
- Chromosome distribution

**Input**: VCF file or bcftools stats output
**Outputs**: Summary plots and statistics

In [ ]:
# Parameters
QUAL_THRESHOLD = 30
DEPTH_MIN = 10
FIGURE_DPI = 150

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from galaxy_notebook import get, put, list_inputs

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = FIGURE_DPI

In [ ]:
# List available inputs
print("Available inputs:")
for inp in list_inputs():
    print(f"  [{inp['index']}] {inp['name']}")

In [ ]:
# Parse VCF file
def parse_vcf(filepath):
    """Parse VCF file into DataFrame."""
    variants = []
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            fields = line.strip().split('\t')
            if len(fields) >= 8:
                chrom, pos, vid, ref, alt, qual, filt, info = fields[:8]
                
                # Determine variant type
                if len(ref) == len(alt) == 1:
                    vtype = 'SNP'
                elif len(ref) > len(alt):
                    vtype = 'Deletion'
                elif len(ref) < len(alt):
                    vtype = 'Insertion'
                else:
                    vtype = 'Complex'
                
                # Parse depth from INFO if available
                depth = None
                for item in info.split(';'):
                    if item.startswith('DP='):
                        depth = int(item.split('=')[1])
                
                variants.append({
                    'chrom': chrom,
                    'pos': int(pos),
                    'ref': ref,
                    'alt': alt,
                    'qual': float(qual) if qual != '.' else None,
                    'filter': filt,
                    'type': vtype,
                    'depth': depth
                })
    return pd.DataFrame(variants)

# Load VCF
vcf_path = get(0, format='path')
df = parse_vcf(vcf_path)
print(f"Loaded {len(df):,} variants")
df.head()

## 1. Variant Type Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
type_counts = df['type'].value_counts()
colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']
axes[0].pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%', colors=colors)
axes[0].set_title('Variant Types')

# Bar chart
type_counts.plot(kind='bar', ax=axes[1], color=colors[:len(type_counts)])
axes[1].set_title('Variant Count by Type')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('')
for i, v in enumerate(type_counts.values):
    axes[1].text(i, v + 50, f'{v:,}', ha='center')

plt.tight_layout()
put(fig, 'variant_types.png')
plt.show()

## 2. Quality Score Distribution

In [ ]:
qual_values = df['qual'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(qual_values, bins=50, color='#3498db', edgecolor='white', alpha=0.7)
axes[0].axvline(QUAL_THRESHOLD, color='red', linestyle='--', label=f'Threshold ({QUAL_THRESHOLD})')
axes[0].set_xlabel('Quality Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Quality Score Distribution')
axes[0].legend()

# Boxplot by variant type
df_qual = df.dropna(subset=['qual'])
df_qual.boxplot(column='qual', by='type', ax=axes[1])
axes[1].set_title('Quality by Variant Type')
axes[1].set_xlabel('Variant Type')
axes[1].set_ylabel('Quality Score')
plt.suptitle('')

plt.tight_layout()
put(fig, 'quality_distribution.png')
plt.show()

## 3. Transition/Transversion Analysis

In [ ]:
# Calculate Ti/Tv for SNPs
snps = df[df['type'] == 'SNP'].copy()

def classify_mutation(ref, alt):
    transitions = [('A', 'G'), ('G', 'A'), ('C', 'T'), ('T', 'C')]
    if (ref, alt) in transitions:
        return 'Transition'
    else:
        return 'Transversion'

if len(snps) > 0:
    snps['mutation_class'] = snps.apply(lambda x: classify_mutation(x['ref'], x['alt']), axis=1)
    
    ti_count = (snps['mutation_class'] == 'Transition').sum()
    tv_count = (snps['mutation_class'] == 'Transversion').sum()
    ti_tv_ratio = ti_count / tv_count if tv_count > 0 else float('inf')
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # Mutation spectrum
    mutation_types = snps.apply(lambda x: f"{x['ref']}>{x['alt']}", axis=1)
    mutation_counts = mutation_types.value_counts()
    
    colors = ['#3498db' if classify_mutation(m[0], m[-1]) == 'Transition' else '#e74c3c' 
              for m in mutation_counts.index]
    mutation_counts.plot(kind='bar', ax=ax, color=colors)
    ax.set_title(f'Mutation Spectrum (Ti/Tv = {ti_tv_ratio:.2f})')
    ax.set_xlabel('Mutation Type')
    ax.set_ylabel('Count')
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#3498db', label='Transition'),
                       Patch(facecolor='#e74c3c', label='Transversion')]
    ax.legend(handles=legend_elements)
    
    plt.tight_layout()
    put(fig, 'mutation_spectrum.png')
    plt.show()
    
    print(f"Transitions: {ti_count:,}")
    print(f"Transversions: {tv_count:,}")
    print(f"Ti/Tv Ratio: {ti_tv_ratio:.2f}")
else:
    print("No SNPs found for Ti/Tv analysis")

## 4. Chromosome Distribution

In [ ]:
# Sort chromosomes naturally
def sort_chromosomes(chroms):
    def chrom_key(c):
        c = c.replace('chr', '')
        if c.isdigit():
            return (0, int(c))
        return (1, c)
    return sorted(chroms, key=chrom_key)

chrom_counts = df['chrom'].value_counts()
sorted_chroms = sort_chromosomes(chrom_counts.index.tolist())
chrom_counts = chrom_counts.reindex(sorted_chroms)

fig, ax = plt.subplots(figsize=(14, 6))
chrom_counts.plot(kind='bar', ax=ax, color='#3498db', edgecolor='white')
ax.set_title('Variants per Chromosome')
ax.set_xlabel('Chromosome')
ax.set_ylabel('Variant Count')
plt.xticks(rotation=45)

plt.tight_layout()
put(fig, 'chromosome_distribution.png')
plt.show()

## 5. Summary Statistics

In [ ]:
# Calculate summary statistics
total = len(df)
passed = (df['filter'] == 'PASS').sum() if 'filter' in df.columns else 'N/A'
snp_count = (df['type'] == 'SNP').sum()
indel_count = (df['type'].isin(['Insertion', 'Deletion'])).sum()

qual_median = df['qual'].median() if df['qual'].notna().any() else 'N/A'

summary = f"""
# Variant Analysis Summary

## Overview
- Total variants: {total:,}
- PASS filter: {passed:,}
- SNPs: {snp_count:,} ({snp_count/total*100:.1f}%)
- Indels: {indel_count:,} ({indel_count/total*100:.1f}%)

## Quality Metrics
- Median quality score: {qual_median}
- Variants above Q{QUAL_THRESHOLD}: {(df['qual'] >= QUAL_THRESHOLD).sum():,}

## Ti/Tv Analysis
- Ti/Tv Ratio: {ti_tv_ratio:.2f if 'ti_tv_ratio' in dir() else 'N/A'}

## Chromosomes
- Chromosomes with variants: {df['chrom'].nunique()}
"""

print(summary)
put(summary, 'variant_summary.md')

In [ ]:
print("\n=== Analysis Complete ===")
print("Generated outputs:")
print("  - variant_types.png")
print("  - quality_distribution.png")
print("  - mutation_spectrum.png")
print("  - chromosome_distribution.png")
print("  - variant_summary.md")